# T1.1 end-to-end smoke: eval-gated retrain on BEIR SciFact

Validates the new `vstash.retrain.retrain()` entry point end-to-end:
1. Ingests ~1000 SciFact docs into a throwaway vstash store.
2. Runs `retrain()` with default min_gain=0.0.
3. Reports baseline NDCG@10, final NDCG@10, delta, and gate outcome.

Designed to be run on Colab where sentence-transformers training works reliably.

In [ ]:
# Cell 1: Setup -- branch feat/retrain-tier1 ships the new retrain() orchestrator
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch feat/retrain-tier1 https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

In [ ]:
# Cell 2: Download SciFact and ingest ~1000 docs into a fresh vstash store
import os
import sys
import shutil

sys.path.insert(0, "/content/vstash")

from experiments.beir_benchmark import download_beir, load_beir
from sentence_transformers import SentenceTransformer
from vstash.store import VstashStore

BASE_MODEL = "BAAI/bge-small-en-v1.5"
STORE_PATH = "/tmp/retrain_t11_scifact.db"
OUTPUT_PATH = "/content/retrained_model"

# Clean slate
for p in (
    STORE_PATH,
    STORE_PATH + "-wal",
    STORE_PATH + "-shm",
    OUTPUT_PATH,
    OUTPUT_PATH + ".candidate",
):
    if os.path.isdir(p):
        shutil.rmtree(p)
    elif os.path.isfile(p):
        os.remove(p)

cache = download_beir("scifact")
corpus, queries, qrels = load_beir(cache)
doc_ids = list(corpus.keys())[:1000]
print(f"loaded {len(corpus)} docs, ingesting first {len(doc_ids)}")

# Embed once with the base model, ingest into vstash.
model = SentenceTransformer(BASE_MODEL)
texts = [(corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip() for d in doc_ids]
vecs = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print(f"embedded {len(texts)} chunks, dim={vecs.shape[1]}")

store = VstashStore(STORE_PATH, embedding_dim=int(vecs.shape[1]))
for doc_id, text, vec in zip(doc_ids, texts, vecs):
    store.add_document(
        path=f"scifact://{doc_id}",
        title=corpus[doc_id].get("title", "")[:80] or doc_id,
        chunks=[text],
        embeddings=[list(map(float, vec))],
    )

stats = store.stats()
print(f"vstash store: {stats.documents} docs, {stats.chunks} chunks")

In [ ]:
# Cell 3: Run the eval-gated retrain pipeline end-to-end
import time
from vstash.retrain import retrain as run_retrain

t0 = time.perf_counter()
result = run_retrain(
    store,
    base_model=BASE_MODEL,
    output_path=OUTPUT_PATH,
    max_queries=800,  # cap pseudo-queries so the run finishes in a few minutes
    epochs=2,
    lr=3e-6,
    batch_size=64,
    eval_fraction=0.15,
    eval_noise_size=500,
    min_gain=0.0,  # default: reject anything that does not beat baseline
)
elapsed = time.perf_counter() - t0
print(f"\nretrain() finished in {elapsed:.1f}s")
print(f"RetrainResult: {result}")

In [ ]:
# Cell 4: Pretty-printed report
import json
from pathlib import Path


def fmt_metrics(m):
    if m is None:
        return "    (not computed)"
    return (
        f"    n_queries:  {m.n_queries}\n"
        f"    NDCG@10:    {m.ndcg_at_10:.4f}\n"
        f"    MRR:        {m.mrr:.4f}\n"
        f"    Hit@10:     {m.hit_at_10:.4f}"
    )


print("=" * 60)
print("T1.1 end-to-end smoke: SciFact 1k subset")
print("=" * 60)
print(f"n_pairs:       {result.n_pairs}")
print(f"gated_out:     {result.gated_out}")
print(f"min_gain:      {result.min_gain:+.4f}")
print(f"output_path:   {result.output_path}")
print()
print("Baseline:")
print(fmt_metrics(result.baseline))
print()
print("Final:")
print(fmt_metrics(result.final))
print()
if result.baseline is not None and result.final is not None:
    delta = result.delta_ndcg
    print(f"Delta NDCG@10: {delta:+.4f}  ({delta * 100:+.2f}%)")

# Surface training_meta.json so we can eyeball the numbers persisted to disk.
meta_path = Path(result.output_path or (OUTPUT_PATH + ".candidate")) / "training_meta.json"
if meta_path.exists():
    print()
    print("training_meta.json:")
    print(json.dumps(json.loads(meta_path.read_text()), indent=2))